# Front End Simulation for QICK-DAWG Compiler

In [12]:
"""
Demo: mock sequences rendered through the simulation backend.

  1. Coarse Rabi    — a coarse-mode drive pulse whose LENGTH is the swept
                      quantity
  2. Amplitude Rabi - A fine-time pulse shape whose amplitude is swept
  2. Ramsey         — swept fine-time delay between two shaped pi/2 pulses
  3. XY8            — fixed repeat() with a phase Pattern on the pulse default

No board needed: units falls back to SoftSocCfg automatically (or swap in
the real one with units.set_soccfg(qd.soccfg)).

Run:  python demo_simulate.py
"""

import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import numpy as np

for candidate in (Path.cwd(), *Path.cwd().parents):
    src_dir = candidate / "src"
    if (src_dir / "qickdawg").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    from qickdawg.compiler import units
    units.get_soccfg()
print("soccfg in use:", type(units.get_soccfg()).__name__, "\n")

from qickdawg.compiler.ir import Sequence, sweep, repeat, play, delay
from qickdawg.compiler.parameters import Parameter
from qickdawg.compiler.pulses import DefinePulse, Shape
from qickdawg.compiler.sweep_axis import LinearSweepAxis, Pattern, RepeatAxis
from qickdawg.compiler.simulate import simulate, sweep_axes, format_axis_point


soccfg in use: SoftSocCfg 



Pulse shape definitions:

In [30]:
F_DRIVE = 100e6

def gaussian(t, sigma):
    return np.exp(-0.5 * ((t - t[-1] / 2) / sigma) ** 2)

gauss512 = Shape.from_func(gaussian, Parameter.constant("plen", 512, "ftsamp"),
                           sigma=512 / 6)
pi2 = DefinePulse("pi/2", amplitude=1.9, phase=0.0, frequency=F_DRIVE,
                  channel=0, shape=gauss512)

# Rabi Example

In [35]:
pulse_dur = LinearSweepAxis.create("pulse_dur", 16, 96, unit="ns", step=4,
                              kind="coarse_time")
drive = DefinePulse("drive", amplitude=1.8, phase=0.0, frequency=F_DRIVE,
                    channel=0, length_coarse=pulse_dur)

with Sequence() as rabi:
    with sweep(pulse_dur):
        play(drive)
        # readout at a fixed slot past the longest drive (96 treg + margin)
        delay(Parameter.constant("wait", 120, "treg"), channel=1)
        # TODO: Add readout

print("Rabi sweep axes:",[(ax.name, ax.kind, ax.num_steps) for ax in sweep_axes(rabi)])

for k in (0, 2, 5):
    r = simulate(rabi, sweep_indices={pulse_dur: k})
    d = next(c for c in r.pulses if c.name == "drive")
    dur = d.pulse.end_ftsamp - d.pulse.start_ftsamp
    print(f"  {format_axis_point(pulse_dur, k, r.unit_model)} -> "
          f"drive duration {dur} ftsamp ({dur // 16} treg)")
    r.plot(save=f"rabi_len_k{k}.png")
print(r.describe(), "\n")

Rabi sweep axes: [('pulse_dur', 'coarse_time', 24)]
  pulse_dur=5 treg (0.01628 us) -> drive duration 80 ftsamp (5 treg)
  pulse_dur=7 treg (0.02279 us) -> drive duration 112 ftsamp (7 treg)
  pulse_dur=10 treg (0.03255 us) -> drive duration 160 ftsamp (10 treg)
Sweep point: pulse_dur=10 treg (0.03255 us) [treg idx 5]
Timeline: 160 ftsamp (0.033 us), 1 shot(s)
  shot0 seg1 ch0:      drive @ [0, 160) ftsamp  f=100.000 MHz  ph=0.0 deg  amp=1.80 



## Ramsey Example

In [32]:
tau = LinearSweepAxis.create("tau", 20, 600, unit="ns", step=10)   # fine kind

with Sequence() as ramsey:
    with sweep(tau):
        play(pi2)
        delay(tau, channel=0)
        play(pi2)
        delay(Parameter.constant("wf_wait", 60, "ns"), channel=1) # explicit declaration of wait time
        # TODO: Add readout

res = simulate(ramsey, sweep_indices={tau: 15})
print(res.describe(), "\n")
res.plot(save="ramsey_example.png")


Sweep point: tau=0.1695 us [ftsamp idx 15]
Timeline: 1857 ftsamp (0.378 us), 1 shot(s)
  shot0 seg1 ch0:       pi/2 @ [0, 512) ftsamp  f=100.000 MHz  ph=0.0 deg  amp=1.90
  shot0 seg1 ch0:       pi/2 @ [1345, 1857) ftsamp  f=100.000 MHz  ph=0.0 deg  amp=1.90 



<Figure size 1100x220 with 1 Axes>

## CPMGXY8 Example

In [33]:
# Linear sweep on Number of pi pulses
N = LinearSweepAxis.create("N_pi", 3, 50, unit="count", step=3)
# Define a number of repetitions
i = RepeatAxis("xy8", bound=N)

# Parameter decleration based on a pattern
xy8 = Pattern.create([0, 90, 0, 90, 90, 0, 90, 0], unit="deg", axis=i)
ending_pi2 = Pattern.create([270, 270, 90, 90, 270, 90, 90, 270], unit="deg", axis=N)

# Default pulse decleration (parameters can be overwritten for instances)
pi_xy8 = DefinePulse("pi", amplitude=1.0, phase=xy8, frequency=F_DRIVE,
                     channel=0, shape=Shape.square(100e-9))
pi2 = DefinePulse("pi/2", amplitude=1.0, phase=0.0, frequency=F_DRIVE,
                  channel=0, shape=Shape.square(50e-9))

tau = Parameter.constant("gap", 406.6, "ns") # Explicit constant parameter decleration

with Sequence() as cpmg:
    with sweep(N):
        delay(500e-9, channel=0) # Implicit parameter declaration (in base SI units, (seconds))
        play(pi2)
        with repeat(i):
            delay(tau, channel=0)
            play(pi_xy8)
            delay(tau, channel=0)
        play(pi2, phase=ending_pi2) # override pulse phase parameter (derive from pattern)
        # TODO: Add readout

res3 = simulate(cpmg, sweep_indices={N: 15})
phases = [f"{c.pulse.phase:.0f}" for c in res3.pulses if c.name == "pi"]
print("XY8 phases:", phases)
res3.plot(save="cpmgxy8_example.png")

XY8 phases: ['0', '90', '0', '90', '90', '0', '90', '0', '0', '90', '0', '90', '90', '0', '90', '0', '0', '90', '0', '90', '90', '0', '90', '0', '0', '90', '0', '90', '90', '0', '90', '0', '0', '90', '0', '90', '90', '0', '90', '0', '0', '90', '0', '90', '90', '0', '90', '0']


<Figure size 1100x220 with 1 Axes>